# Hyperparameter Inverse PINN
## Lambda Data Sweep


In [1]:
# Standard library imports
import os
import time
from pathlib import Path

# Data manipulation and mathematics
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt

# Deep learning framework (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim

# Sweep
import itertools


In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/egil10/fys5429.git"
    repo_dir = "/content/fys5429"

    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    else:
        !git -C {repo_dir} pull

    os.chdir(os.path.join(repo_dir, "code", "notebooks"))

print(f"Working directory: {os.getcwd()}")


Working directory: c:\Users\ofurn\Dokumenter\Github\fys5429\code\notebooks


### Colab setup


In [3]:
data_path = Path("..") / "data" / "generated" / "inv_collocation.parquet"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = Path("/content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/eda")
else:
    out_dir = Path("..") / "plots" / "eda"

out_dir.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {out_dir}")


Plots will be saved to: ..\plots\eda


In [4]:
import sys
sys.path.insert(0, "../scripts")
from invpinn import INVPINN
from train_inv import train_inv_pinn


### Global parameters


In [5]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
torch.backends.cudnn.benchmark = True

# Domain (same as forward)
S_max = 300.0
T_max = 1.0
K = 100.0
r = 0.05
v_min = 0.01
v_max = 1.0
v0 = 0.04

# NN architecture (proven from forward sweeps)
HIDDEN_LAYERS = 3
NEURONS_PER_LAYER = 256
ACTIVATION = 'tanh'
LEARNING_RATE = 5e-3

# Physics lambdas (proven from forward sweeps)
LAMBDA_PDE = 10.0
LAMBDA_IC  = 10.0
LAMBDA_BC  = 5.0

# NEW: Sweep over lambda_data
LAMBDA_DATA_VALUES = [1.0, 5.0, 10.0, 20.0, 50.0]

# Intentionally wrong initial guesses for the Heston parameters
KAPPA_INIT = 1.0    # True: 2.0
THETA_INIT = 0.1    # True: 0.04
XI_INIT    = 0.5    # True: 0.3
RHO_INIT   = 0.0    # True: -0.7

SWEEP_EPOCHS = 5000


Using device: cpu


In [6]:
if not data_path.exists():
    raise FileNotFoundError(f"Data not found at {data_path}. Run generate_inv.ipynb first.")
else:
    df_all = pd.read_parquet(data_path)
    print(f"Loaded data from {data_path}")

def extract_tensors(df_subset):
    S = torch.tensor(df_subset['S'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    v = torch.tensor(df_subset['v'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    tau = torch.tensor(df_subset['tau'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    return S, v, tau

# Interior (PDE)
df_interior = df_all[df_all['point_type'] == 'interior']
S_in, v_in, tau_in = extract_tensors(df_interior)

# IC 
df_ic = df_all[df_all['point_type'] == 'initial_condition']
S_ic, v_ic, tau_ic = extract_tensors(df_ic)

# BC (S=0 only, as per the fix from forward solver)
df_bc = df_all[df_all['point_type'] == 'boundary_S_lower']
S_bc, v_bc, tau_bc = extract_tensors(df_bc)

# Market data (NEW)
df_data = df_all[df_all['point_type'] == 'market_data']
S_data, v_data, tau_data = extract_tensors(df_data)
V_data = torch.tensor(df_data['V_data'].values, dtype=torch.float32).view(-1, 1).to(device)

print(f"Interior: {len(S_in)}, IC: {len(S_ic)}, BC: {len(S_bc)}, Data: {len(S_data)}")


FileNotFoundError: Data not found at ..\data\generated\inv_collocation.parquet. Run generate_inv.ipynb first.

In [ ]:
sweep_results = []
start_time = time.time()
total_runs = len(LAMBDA_DATA_VALUES)

header = f"{'#':>3} | {'λ_data':>8} | {'PDE':>10} {'IC':>10} {'BC':>10} {'Data':>10} | {'κ':>6} {'θ':>6} {'ξ':>6} {'ρ':>6} | {'Time':>5}"
print(header)
print("─" * len(header))

for i, lam_data in enumerate(LAMBDA_DATA_VALUES):
    run_start = time.time()

    result = train_inv_pinn(
        S_in, v_in, tau_in,
        S_ic, v_ic, tau_ic,
        S_bc, v_bc, tau_bc,
        S_data, v_data, tau_data, V_data,
        r, K, device,
        LAMBDA_PDE, LAMBDA_IC, LAMBDA_BC, lam_data, SWEEP_EPOCHS,
        lr=LEARNING_RATE, hidden_layers=HIDDEN_LAYERS,
        neurons=NEURONS_PER_LAYER, activation=ACTIVATION,
        kappa_init=KAPPA_INIT, theta_init=THETA_INIT,
        xi_init=XI_INIT, rho_init=RHO_INIT
    )

    result['lambda_data'] = lam_data
    sweep_results.append(result)

    run_sec = time.time() - run_start
    print(f"{i+1:>3} | {lam_data:>8.1f} | "
          f"{result['final_pde']:>10.4f} {result['final_ic']:>10.4f} "
          f"{result['final_bc']:>10.6f} {result['final_data']:>10.4f} | "
          f"{result['final_kappa']:>6.3f} {result['final_theta']:>6.4f} "
          f"{result['final_xi']:>6.3f} {result['final_rho']:>6.3f} | "
          f"{run_sec:>5.0f}s")

elapsed = time.time() - start_time
print("─" * len(header))
print(f"Sweep complete: {len(sweep_results)} runs in {int(elapsed//60)}m {int(elapsed%60):02d}s")


In [ ]:
df_sweep = pd.DataFrame([{
    'lambda_data': r['lambda_data'],
    'pde_loss': r['final_pde'],
    'data_loss': r['final_data'],
    'total_loss': r['final_total'],
    'kappa': r['final_kappa'],
    'theta': r['final_theta'],
    'xi': r['final_xi'],
    'rho': r['final_rho'],
} for r in sweep_results])

df_sweep = df_sweep.sort_values('data_loss')
print(df_sweep.to_string(index=False))


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Google Drive flushed and unmounted safely.")
